# Пробный прогон T-lite: эпизод целиком

Исходная T-lite-it-2.1 (без обучения) с системным промптом `prompts.py`: промпт → генерация → вызовы Википедии →
ответ → разбор DSL → PPTX. Смотрим, держит ли модель формат без примера в промпте, ищет ли сама, сколько занимает
эпизод по токенам, времени и памяти. Запуск — на сервере (V100 32 ГБ, float16).

Результаты — `runs/tlite_pilot/<время>/`: для каждого задания весь эпизод текстом (`<id>.txt`), ответ для парсера
(`<id>_answer.txt`), презентация (`<id>.pptx`) и сводка `summary.json`.

In [2]:
import dataclasses, json, os, sys, time
from pathlib import Path
os.environ["CUDA_VISIBLE_DEVICES"] = "4"
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))  # если пакет не установлен через pip install -e .
import torch
from presentation_rl.dsl import parse
from presentation_rl.environment import build_prompt, run_episode
from presentation_rl.model_loader import hf_generator, load_model
from presentation_rl.prompts import system_prompt
from presentation_rl.rendering import build_presentation, pick_theme
from presentation_rl.tasks import load_tasks
from presentation_rl.tools.cache import WikiCache
from presentation_rl.tools.wiki import WikiClient
from presentation_rl.tools.wiki_tools import WikiTools

print(torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
CONTACT = os.environ.get("WIKI_CONTACT", "https://github.com/doshik722")  # или впишите сюда почту / ссылку на проект
cache = WikiCache(ROOT / "data" / "sources" / "wiki_cache", WikiClient(contact=CONTACT))
OUT = ROOT / "runs" / "tlite_pilot" / time.strftime("%Y%m%d_%H%M")
OUT.mkdir(parents=True, exist_ok=True)

2.14.1+cu126 Tesla V100-SXM3-32GB (7, 0)


In [3]:
start = time.time()
model, tokenizer = load_model("t-tech/T-lite-it-2.1", dtype="float16")
generate = hf_generator(model, tokenizer, temperature=0.9)
print(f"загрузка {time.time() - start:.0f} с, память {torch.cuda.memory_allocated() / 2**30:.1f} ГБ")

/home/ryzhkova/venvs/diplom/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:06<00:00, 63.32it/s]


загрузка 188 с, память 15.3 ГБ


In [4]:
# Четыре задания validation и четыре сочетания полей «Поиск» и «Картинки».
tasks = {t.id: t for t in load_tasks(ROOT / "data" / "tasks" / "tasks.json", split="validation")}
RUNS = [("t367", True, True), ("t366", True, False), ("t369", False, True), ("t370", False, False)]

In [5]:
summary = []
for task_id, search, images in RUNS:
    request = dataclasses.replace(tasks[task_id].request, search=search, images=images)
    tools = WikiTools(cache) if search else None
    prompt_ids = build_prompt(tokenizer, system_prompt(request), request)
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    episode = run_episode(prompt_ids, tokenizer, generate, tools, max_new_tokens=8000)
    seconds = time.time() - start
    deck = parse(episode.answer)
    report = build_presentation(deck, OUT / f"{task_id}.pptx", pick_theme(request.theme, request.text),
                                sources=tools.source if tools else None, show_sources=request.sources)
    (OUT / f"{task_id}.txt").write_text(tokenizer.decode(episode.ids), encoding="utf-8")
    (OUT / f"{task_id}_answer.txt").write_text(episode.answer, encoding="utf-8")
    row = {
        "id": task_id, "search": search, "images": images, "finish": episode.finish,
        "prompt_tokens": len(episode.prompt_ids), "generated_tokens": episode.generated_tokens,
        "total_tokens": len(episode.ids), "calls": [f"{c['name']}({c['argument']})" for c in episode.calls],
        "parse_errors": len(deck.errors), "error_codes": sorted({e.code for e in deck.errors}),
        "slides": report.slide_count, "render": report.status_counts(), "seconds": round(seconds),
        "peak_memory_gb": round(torch.cuda.max_memory_allocated() / 2**30, 1),
    }
    summary.append(row)
    print(json.dumps(row, ensure_ascii=False))
(OUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1), encoding="utf-8")

{"id": "t367", "search": true, "images": true, "finish": "stop", "prompt_tokens": 2183, "generated_tokens": 1766, "total_tokens": 3949, "calls": [], "parse_errors": 8, "error_codes": ["bad_value", "outside_slide"], "slides": 9, "render": {"ok": 16, "placeholder": 5}, "seconds": 88, "peak_memory_gb": 15.8}
{"id": "t366", "search": true, "images": false, "finish": "stop", "prompt_tokens": 2198, "generated_tokens": 296, "total_tokens": 2494, "calls": [], "parse_errors": 8, "error_codes": ["outside_slide"], "slides": 0, "render": {}, "seconds": 14, "peak_memory_gb": 15.8}
{"id": "t369", "search": false, "images": true, "finish": "stop", "prompt_tokens": 1599, "generated_tokens": 1615, "total_tokens": 3214, "calls": [], "parse_errors": 51, "error_codes": ["outside_slide"], "slides": 0, "render": {}, "seconds": 92, "peak_memory_gb": 15.7}
{"id": "t370", "search": false, "images": false, "finish": "stop", "prompt_tokens": 1616, "generated_tokens": 311, "total_tokens": 1927, "calls": [], "pars

1289

In [6]:
# Первый эпизод с поиском целиком — как его видела модель (без системного промпта).
text = (OUT / "t367.txt").read_text(encoding="utf-8")
print(text[text.index("<|im_start|>user"):])

<|im_start|>user
Напиши презентацию на 10 слайдов на тему систем счисления в информатике

Слайдов: 10
Поиск: есть
Картинки: есть<|im_end|>
<|im_start|>assistant
<think>

</think>

#PLAN: понятие → позиционные системы → преобразование между системами → арифметика → компьютерные реализации → приложения → история → сравнение систем → выводы
#TITLE: Системы счисления в информатике
#TEXT: Системы счисления — способ представления чисел с помощью знаков (цифр). В информатике важны позиционные системы: десятичная, двоичная, восьмеричная, шестнадцатеричная. Двоичная — основа работы всех компьютеров.
#IMAGE: двоичная система счисления с примерами: 0, 1, 10, 11, 100
#NOTES: Рассказать, что двоичная система используется потому, что легко реализуется в виде двух состояний: есть ток/нет тока.

#SLIDE
#TITLE: Позиционные и непозиционные системы
#TEXT: В позиционных системах значение цифры зависит от её позиции (разрядов). Например, в числе 123 цифра 2 — это 20. Непозиционные (рукописные) теряют важно